# TP 1 — LDA/QDA y optimización matemática de modelos (resuelto)

**Materia:** Análisis Matemático para IA (AMIA) — CEIA, FIUBA
**Alumno:** Gustavo Varela

Notebook resuelto del TP. Las clases base (`QDA`, `TensorizedQDA`, `QDA_Chol1/2/3`)
son las provistas por la cátedra y viven en `base/`. Las **4 clases nuevas**
(`FasterQDA`, `EfficientQDA`, `TensorizedChol`, `EfficientChol`) están en esos
mismos módulos y se muestran más abajo con `inspect.getsource`.

> **Notación:** $k$ = nº de clases, $n$ = nº de observaciones, $p$ = nº de features.
> Las matrices de datos van traspuestas: $X \in \mathbb{R}^{p \times n}$ (features en filas).

Referencia teórica: *Mathematics for Machine Learning* (Deisenroth, Faisal & Ong) —
§6.5 (Gaussiana), §4.1 (determinante), §4.3 (Cholesky), §3.2 (productos internos / def. positivas).

## 0. Reproducibilidad — versiones

In [1]:
import sys, numpy, scipy, sklearn
print("Python :", sys.version.split()[0])
print("NumPy  :", numpy.__version__)
print("SciPy  :", scipy.__version__)
print("sklearn:", sklearn.__version__)

Python : 3.12.13
NumPy  : 2.3.1
SciPy  : 1.16.0
sklearn: 1.7.0


In [2]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
from inspect import getsource

from base.qda import QDA, TensorizedQDA, FasterQDA, EfficientQDA
from base.cholesky import (QDA_Chol1, QDA_Chol2, QDA_Chol3,
                           TensorizedChol, EfficientChol)
from utils.datasets import get_wine_dataset, get_letters_dataset, label_encode, split_transpose
from utils.bench import Benchmark, cv_accuracy

pd.set_option("display.width", 200, "display.max_columns", 20)

## 1. Preguntas sobre el código base (Q3–Q7)



---

> **Q3 · ¿Para qué sirve `bincount`?**

**La idea.** Todo clasificador bayesiano arranca preguntándose, *antes de mirar las features*, qué tan probable es cada clase: las probabilidades a priori $\pi_j$. La forma más honesta de estimarlas a partir de los datos es **contar**: si en la muestra el 33 % son de la clase A, esa es nuestra mejor apuesta para $\pi_A$. `bincount` es, simplemente, la herramienta que cuenta.

**El detalle.** `np.bincount(y)` devuelve $[n_0,\dots,n_{k-1}]$ (cuántas veces aparece cada etiqueta). Dividido por $n$ da las frecuencias relativas $\hat\pi_j = n_j/n$ —que resultan ser el estimador de máxima verosimilitud de $\pi_j$— y el `log` final es porque el modelo suma log-probabilidades en lugar de multiplicar probabilidades (más estable numéricamente).

---

> **Q4 · ¿Por qué `_fit_params` va al final de `fit`, después de estimar las a priori?**

**La idea.** Hay un **orden natural de dependencia**. Para estimar los parámetros de cada clase ($\mu_j, \Sigma_j$), el modelo primero necesita saber *cuántas clases hay y cuáles son*. Y esa información ya quedó resuelta al calcular las a priori. Además, las a priori son la única pieza que el usuario podría querer **fijar a mano** en vez de estimar de los datos, así que tiene sentido resolverlas primero y dejar lo "puramente de datos" para el final.

**El detalle.** `_fit_params` itera con `len(self.log_a_priori)` para recorrer las clases. Ese atributo se crea recién en `_estimate_a_priori`; si moviéramos `_fit_params` antes, todavía no existiría → `AttributeError`.

---

> **Q5 · ¿Por qué el `flatten` y no directamente `X[:, y==idx]`?**

**La idea.** Estamos partiendo los datos por clase: necesitamos señalar *qué observaciones* (columnas de $X$) pertenecen a la clase $j$. Esa "lista de columnas elegidas" tiene que ser un vector plano, una marca por observación.

**El detalle.** `y` viene como `(1, n)` (vector fila), así que `y==idx` da una máscara **2-D** `(1, n)`. Para indexar las columnas de $X$ (que es `(p, n)`) hace falta una máscara **1-D** de largo $n$. `y.flatten()` la aplana a `(n,)` y recién ahí `X[:, y.flatten()==idx]` selecciona bien.

---

> **Q6 · ¿Por qué `bias=True` en `np.cov`?**

**La idea.** Hay dos maneras clásicas de estimar una covarianza: la **insesgada** (divide por $n-1$, corrige el sesgo en muestras chicas) y la de **máxima verosimilitud** (divide por $n$). Todo el armado teórico de QDA es de máxima verosimilitud puro —todos los parámetros salen de maximizar la verosimilitud—, así que por **coherencia** la covarianza tiene que estimarse con el mismo criterio.

**El detalle.** `bias=False` (default) usa $n-1$; `bias=True` usa $n$, que es justamente el estimador MV $\hat\Sigma_j$ que aparece en la derivación del TP.

---

> **Q7 · ¿Qué hace `axis=1`? ¿Por qué no `axis=0`?**

**La idea.** La media de una clase es su **centro de gravedad** en el espacio de features: un promedio por cada variable, calculado a lo largo de todas las observaciones de esa clase. La pregunta es solo "¿en qué dirección de la matriz viven las observaciones?".

**El detalle.** $X$ es `(p, n)`: las features están en las filas y las observaciones en las columnas. Promediar "a lo largo de las observaciones" = promediar sobre las columnas = `axis=1`, dando `(p, 1)`. `axis=0` promediaría sobre las features (mezclaría variables distintas, sin sentido). `keepdims=True` mantiene la forma `(p, 1)` para el *broadcasting* posterior contra $x$.

## 2. Tensorización

> **La idea de fondo de toda esta sección.** El modelo no cambia: la predicción es siempre "elegir la clase que maximiza log-priori + log-verosimilitud". Lo único que cambia es *cómo se computa*. El `QDA` original recorre con dos `for` anidados (uno sobre observaciones, otro sobre clases) en Python puro, que es lento. Tensorizar = reemplazar esos bucles por operaciones de álgebra matricial que NumPy ejecuta en C, todas de una. Mismo resultado, mucho menos overhead del intérprete.

---

> **P1 · ¿Sobre qué paraleliza `TensorizedQDA`? ¿Clases, observaciones, o ambas?**

**La idea.** Da un primer paso, no el completo: paraleliza **solo sobre las $k$ clases**. Sigue prediciendo de a una observación por vez (el `for` sobre observaciones de `predict` queda intacto), pero para cada observación calcula las $k$ verosimilitudes de un saque, en vez de recorrer las clases con una *list-comprehension*.

**El detalle.** El loop sobre clases de `QDA` se reemplaza apilando las matrices por clase en un tensor y operando sobre todo el "stack" a la vez. **No** toca el loop sobre las $n$ observaciones; eso queda para `FasterQDA`.

---

> **P2 · Analizar los shapes de `tensor_inv_cov` y `tensor_means` y explicar por qué `TensorizedQDA` predice lo mismo que `QDA`.**

**La idea.** Apilar las $k$ matrices/medias en una dimensión extra permite que una sola multiplicación matricial "batcheada" calcule, en paralelo, la misma forma cuadrática que `QDA` calculaba clase por clase. El truco es puramente de *bookkeeping* de dimensiones: el número que sale al final es idéntico.

**El detalle.**
- `tensor_inv_cov`: `(k, p, p)` — las $k$ matrices $\Sigma_j^{-1}$ apiladas.
- `tensor_means`: `(k, p, 1)` — los $k$ vectores $\mu_j$ apilados.

Para una observación $x$ de shape `(p, 1)`:

| paso | operación | shape |
|---|---|---|
| centrado | `x - tensor_means` (broadcast `(p,1)`−`(k,p,1)`) | `(k, p, 1)` |
| traspuesta | `.transpose(0,2,1)` | `(k, 1, p)` |
| $\cdot\,\Sigma^{-1}$ | `@ tensor_inv_cov` | `(k, 1, p)` |
| $\cdot\,(x-\mu)$ | `@ unbiased` | `(k, 1, 1)` |
| `flatten` | | `(k,)` |

Ese `(k,)` contiene, por clase, la forma cuadrática $(x-\mu_j)^T \Sigma_j^{-1} (x-\mu_j)$ —exactamente lo que `QDA` obtenía en su loop—. El término $\tfrac12\log|\Sigma_j^{-1}|$ se computa análogo y se suma la log-priori. Mismo número; cambió la forma de llegar a él.

> **P3 · Implementar `FasterQDA` eliminando el `for` de `predict`.**

**La idea.** `TensorizedQDA` ya sacó el loop sobre clases; falta el otro, el que recorre las $n$ observaciones. La propuesta es procesar **toda la matriz $X \in \mathbb{R}^{p\times n}$ de una sola pasada**, paralelizando a la vez sobre clases *y* observaciones. Hereda de `TensorizedQDA` y solo reescribe `predict`. (El código está más abajo.)

---

> **P4 · Mostrar dónde aparece la matriz de $n\times n$.**

**La idea.** Hacer todo "de una" tiene un costo escondido. Si calculamos la forma cuadrática batcheando las $n$ observaciones igual que antes batcheábamos las clases, el álgebra nos obliga a computar **todos los productos cruzados entre observaciones**: no solo $(x_i-\mu)^T\Sigma^{-1}(x_i-\mu)$ (lo que queremos), sino también $(x_i-\mu)^T\Sigma^{-1}(x_j-\mu)$ para todo par $i\neq j$. Eso es una matriz $n\times n$ por clase, de la que **solo usamos la diagonal** y tiramos el resto.

**El detalle.** En `unbiased.transpose(0,2,1) @ tensor_inv_cov @ unbiased` con `unbiased` de shape `(k, p, n)`, el resultado es `(k, n, n)`. La diagonal de cada bloque son las $n$ formas cuadráticas útiles; el resto, interacciones cruzadas descartadas. Es $O(n^2)$ en cómputo y memoria — innecesario, y es exactamente lo que `EfficientQDA` (P5–P6) va a evitar.

In [3]:
print(getsource(FasterQDA))

class FasterQDA(TensorizedQDA):
    """P3) Elimina el ciclo `for` de `predict`: paraleliza sobre clases Y
    observaciones simultáneamente.

    A diferencia de `TensorizedQDA` (que paraleliza sobre las k clases pero
    sigue iterando observación por observación), acá pasamos toda la matriz
    X (p, n) de una sola vez.

    P4) El precio es que aparece explícitamente una matriz de n x n: al hacer
    `unbiased.transpose(0,2,1) @ inv_cov @ unbiased` con unbiased de shape
    (k, p, n), el resultado es (k, n, n). Solo nos interesa su diagonal
    (las n formas cuadráticas (x_i - mu)^T Sigma^-1 (x_i - mu)); el resto de
    la matriz son "interacciones cruzadas" entre observaciones distintas que
    se descartan. Es O(n^2) en cómputo y memoria, innecesariamente.
    """

    def predict(self, X):
        # X: (p, n)
        # unbiased[k] = X - mu_k  ->  (k, p, n) por broadcasting
        unbiased = X[np.newaxis, :, :] - self.tensor_means

        # (k, n, p) @ (k, p, p) @ (k, p, n) -> (

> **P5 · Demostrar que $\operatorname{diag}(A\cdot B) = \sum_{\text{cols}} A \odot B^T = \texttt{np.sum}(A \odot B^T,\ \texttt{axis=1})$.**

**La idea.** La matriz $n\times n$ de P4 es un desperdicio porque calculamos $n^2$ números para quedarnos con $n$. La pregunta es: ¿podemos obtener **solo la diagonal** de un producto $AB$ sin construir $AB$ entero? Sí: cada elemento de la diagonal es un producto punto entre una fila de $A$ y la columna correspondiente de $B$, y eso se consigue multiplicando elemento a elemento (Hadamard) y sumando. Pasamos de armar una matriz $n\times n$ a operar con matrices $n\times p$ — barato.

**El detalle (demostración).** Sea $A\in\mathbb{R}^{n\times p}$, $B\in\mathbb{R}^{p\times n}$. Por definición del producto matricial, el $i$-ésimo elemento de la diagonal es
$$(AB)_{ii} = \sum_{j=1}^{p} A_{ij}\,B_{ji}.$$
El producto de Hadamard con $B^T$ (que es $n\times p$, con $(B^T)_{ij}=B_{ji}$) da
$$(A \odot B^T)_{ij} = A_{ij}\,(B^T)_{ij} = A_{ij}\,B_{ji}.$$
Sumando sobre $j$ (las columnas, `axis=1`):
$$\sum_{j=1}^{p}(A\odot B^T)_{ij} = \sum_{j=1}^{p} A_{ij}B_{ji} = (AB)_{ii} = \operatorname{diag}(AB)_i. \qquad\blacksquare$$
Nunca se materializa la $n\times n$. La forma equivalente $\texttt{np.sum}(A^T\odot B,\ \texttt{axis=0})^T$ sale de trasponer todo.

---

> **P6 · Usar la propiedad para reimplementar la predicción en `EfficientQDA`.**

**La idea.** Aplicamos P5 a la forma cuadrática de QDA. Tomando $A = (x-\mu)^T$ y $B = \Sigma^{-1}(x-\mu)$, la diagonal que buscábamos es directamente "centrar, multiplicar por $\Sigma^{-1}$, y sumar el producto elemento a elemento sobre las features". Mismo resultado que `FasterQDA`, pero sin pasar nunca por la $n\times n$.

**El detalle.**
$$(x-\mu)^T\Sigma^{-1}(x-\mu) = \sum_{p} \big[\,\texttt{unbiased} \odot (\Sigma^{-1}\texttt{unbiased})\,\big],$$
todo en tensores `(k, p, n)`. La memoria pasa de $O(n^2)$ a $O(np)$. (El código está más abajo.)

In [4]:
print(getsource(EfficientQDA))

class EfficientQDA(TensorizedQDA):
    """P6) Reimplementa `FasterQDA` esquivando la matriz de n x n.

    Usa la identidad demostrada en P5:
        diag(A @ B) = np.sum(A * B.T, axis=1)
    con A = unbiased^T (n, p) y B = inv_cov @ unbiased (p, n). Eso es
    equivalente a, directamente, sumar sobre las p features el producto
    elemento a elemento de `unbiased` con `inv_cov @ unbiased`:
        (x-mu)^T Sigma^-1 (x-mu) = sum_p  unbiased * (Sigma^-1 unbiased)
    Nunca se materializa la (n, n): se trabaja con matrices (k, p, n).
    """

    def predict(self, X):
        # X: (p, n)
        unbiased = X[np.newaxis, :, :] - self.tensor_means        # (k, p, n)
        M = self.tensor_inv_cov @ unbiased                        # (k, p, n)

        # diag de la forma cuadrática vía Hadamard + suma sobre p: (k, n)
        quad = np.sum(unbiased * M, axis=1)

        log_dets = 0.5 * np.log(LA.det(self.tensor_inv_cov))[:, np.newaxis]
        log_conditionals = log_dets - 0.5 * quad
      

### Verificación: las 4 variantes de QDA predicen lo mismo

In [5]:
X_full, y_full = get_wine_dataset()
y_enc = label_encode(y_full)
Xtr, Xte, ytr, yte = split_transpose(X_full, y_enc, test_size=0.3, random_state=42)

qda_models = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]
ref = None
print(f'{"model":16s} {"acc":>7s}  igual a QDA?')
for M in qda_models:
    m = M(); m.fit(Xtr, ytr); p = m.predict(Xte)
    if ref is None: ref = p
    acc = (yte.flatten() == p.flatten()).mean()
    print(f'{M.__name__:16s} {acc:7.4f}  {np.array_equal(p, ref)}')

model                acc  igual a QDA?
QDA               0.9815  True
TensorizedQDA     0.9815  True
FasterQDA         0.9815  True
EfficientQDA      0.9815  True


### P7 — Benchmark de las 4 variantes de QDA

In [6]:
b_wine = Benchmark(X_full, y_enc, n_runs=200, warmup=30, mem_runs=30,
                   test_sz=0.3, same_splits=False)
for M in qda_models:
    b_wine.bench(M)

cols = ['test_median_ms', 'test_speedup', 'test_mem_median_mb',
        'test_mem_reduction', 'mean_accuracy']
b_wine.summary(baseline='QDA')[cols].round(4)

Benching params:
Total runs: 260
Warmup runs: 30
Peak Memory usage runs: 30
Running time runs: 200
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,0.6917,1.0000,0.0077,1.0000,0.9826
TensorizedQDA,0.3175,2.1788,0.0121,0.6349,0.9851
FasterQDA,0.0185,37.4322,0.1093,0.0704,0.9848
EfficientQDA,0.0180,38.3400,0.0750,0.1026,0.9856


> **P7 · Comparar las 4 variantes de QDA. ¿Qué se observa? ¿Se condice con lo esperado?**

**Lectura conceptual.** Lo que la tabla cuenta es *dónde estaba realmente el cuello de botella*. No era la aritmética (las cuentas son pocas y chicas): era el **overhead del intérprete de Python** corriendo bucles. Por eso:

- Sacar solo el loop sobre clases (`TensorizedQDA`) ayuda poco (~2×): el loop caro, el de las $n$ observaciones, sigue ahí.
- Sacar *ese* loop (`FasterQDA`/`EfficientQDA`) es lo que dispara el *speedup* (~37×). Confirma que el costo dominante era iterar en Python, no calcular.
- `FasterQDA` y `EfficientQDA` corren prácticamente igual de rápido, pero **no son equivalentes**: la primera paga ese tiempo construyendo una matriz $n\times n$ que la segunda evita. En Wine ($n$ chico) esa diferencia de memoria casi no se nota — el contraste se vuelve dramático en el dataset grande (lo vemos en P13).

**¿Se condice con lo esperado?** Sí. Vectorizar elimina el overhead del bucle, y la versión que aplica la identidad de P5 consigue el mismo tiempo con memoria mínima. La moraleja temprana: *más rápido* y *más eficiente en memoria* no son lo mismo, y conviene apuntar a las dos cosas.

## 3. Cholesky

> **La idea de fondo.** Hasta acá optimizamos la *predicción*. Pero queda un costo que arrastramos en el *entrenamiento*: invertir cada matriz de covarianza $\Sigma_j$. Invertir una matriz general es caro ($O(p^3)$) y numéricamente delicado. Cholesky aprovecha que $\Sigma$ es simétrica y definida positiva para factorizarla como $\Sigma = LL^T$ con $L$ triangular, y trabajar con la triangular es más barato y estable. (MML §4.3.)

---

> **P8 · Si $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo ayuda en la forma cuadrática de QDA?**

**La idea.** Nunca necesitamos $\Sigma^{-1}$ "como objeto"; lo único que hacemos con ella es la forma cuadrática $(x-\mu)^T\Sigma^{-1}(x-\mu)$. Cholesky permite reescribir esa cantidad como **una simple norma al cuadrado** de un vector transformado, que es lo más barato y estable que se puede pedir. Y, de regalo, el log-determinante (el otro término del modelo) sale de mirar la diagonal de $L$.

**El detalle.** Con $A=LL^T$:
$$A^{-1} = (LL^T)^{-1} = L^{-T}L^{-1} = (L^{-1})^T (L^{-1}).$$
Entonces, definiendo $y = L^{-1}(x-\mu)$:
$$(x-\mu)^T\Sigma^{-1}(x-\mu) = \lVert L^{-1}(x-\mu)\rVert^2 = \lVert y\rVert^2 = \texttt{(y**2).sum()}.$$
Y como $\det\Sigma^{-1} = \big(\prod_i (L^{-1})_{ii}\big)^2$, el término del determinante es $\tfrac12\log|\Sigma^{-1}| = \log\prod_i (L^{-1})_{ii}$. Todo a partir de una triangular.

---

> **P9 · ¿En qué se diferencia `QDA_Chol1` de `QDA`, y cómo llega a la predicción?**

**La idea.** Computan lo mismo por dos caminos. `QDA` toma el camino directo y caro: invierte $\Sigma$ entera y arma la forma cuadrática con el producto matricial completo. `QDA_Chol1` toma el atajo de P8: factoriza, invierte solo la triangular, y convierte la forma cuadrática en una norma.

**El detalle.** `QDA_Chol1`:
1. factoriza $\Sigma_j = L_j L_j^T$ (`cholesky(..., lower=True)`);
2. invierte la triangular $L_j$ y guarda $L_j^{-1}$;
3. en `predict`: $y = L^{-1}\,\text{unbiased}$ y devuelve $\log(\prod\operatorname{diag}L^{-1}) - \tfrac12\lVert y\rVert^2$.

Mismo número que `QDA` por la identidad de P8.

---

> **P10 · ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?**

**La idea.** Las tres parten de la misma factorización $\Sigma=LL^T$; la diferencia es una decisión de ingeniería: *¿invertir $L$ de antemano, o resolver el sistema en cada predicción?* Y, si se invierte, *¿con qué rutina?* Son tres puntos distintos en el trade-off entre trabajo-en-el-fit y trabajo-en-el-predict.

**El detalle.**

| | qué guarda | forma cuadrática en `predict` | log-det |
|---|---|---|---|
| **Chol1** | $L^{-1}$ vía `LA.inv` (inversa **genérica** de la triangular) | $y = L^{-1}u$, $\lVert y\rVert^2$ | $+\log\prod\operatorname{diag}L^{-1}$ |
| **Chol2** | $L$ (no invierte) | resuelve $Ly=u$ con `solve_triangular` (forward subst.) | $-\log\prod\operatorname{diag}L$ |
| **Chol3** | $L^{-1}$ vía `dtrtri` (LAPACK **especializado** en triangulares) | $y = L^{-1}u$, $\lVert y\rVert^2$ | $+\log\prod\operatorname{diag}L^{-1}$ |

Detalle fino: en Chol2 el signo del log-det se invierte porque guarda $L$ y no $L^{-1}$ ($\det L^{-1}=1/\det L$). Chol1 y Chol3 hacen lo mismo salvo la rutina de inversión: `dtrtri` explota la estructura triangular, `LA.inv` no.

### P11 — Benchmark de las 7 variantes (4 QDA + 3 Cholesky)

In [7]:
chol_models = [QDA_Chol1, QDA_Chol2, QDA_Chol3]
for M in chol_models:
    b_wine.bench(M)

order7 = ['QDA','TensorizedQDA','FasterQDA','EfficientQDA','QDA_Chol1','QDA_Chol2','QDA_Chol3']
b_wine.summary(baseline='QDA').loc[order7, cols].round(4)

QDA_Chol1 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,0.6917,1.0000,0.0077,1.0000,0.9826
TensorizedQDA,0.3175,2.1788,0.0121,0.6349,0.9851
FasterQDA,0.0185,37.4322,0.1093,0.0704,0.9848
EfficientQDA,0.0180,38.3400,0.0750,0.1026,0.9856
QDA_Chol1,0.3845,1.7989,0.0078,0.9843,0.9831
QDA_Chol2,0.8938,0.7739,0.0080,0.9660,0.9814
QDA_Chol3,0.3885,1.7805,0.0076,1.0099,0.9822


> **P11 · Comparar las 7 variantes. ¿Hay alguna `QDA_Chol` claramente mejor o peor?**

**Lectura conceptual.** La comparación entre las tres Cholesky es, en el fondo, *dónde poner el trabajo*: si lo hacés una vez en el `fit` (precomputar $L^{-1}$) o lo repetís en cada predicción (resolver el sistema).

- `QDA_Chol1` y `QDA_Chol3` precomputan $L^{-1}$, así que en `predict` solo multiplican: quedan parejas (~1.8×). Lo que las separa es la rutina de inversión en el fit (`dtrtri` vs `LA.inv`), diferencia casi invisible con $p$ chico.
- `QDA_Chol2` es **la peor en predicción**, incluso por debajo de `QDA` (~0.78×). El motivo es conceptual, no anecdótico: resuelve un sistema triangular **por cada observación**, y ese costo por-llamada (más el overhead de entrar a `solve_triangular`) no se amortiza. Cambió "invertir una vez" por "resolver $n$ veces", y para muchos `predict` eso es mal negocio.

**Conclusión para lo que sigue.** Si vamos a tensorizar, conviene partir de una variante que **precompute $L^{-1}$** (Chol1/Chol3), porque esa inversa explícita se apila y batchea naturalmente — algo que `solve_triangular` no permite. Elijo `QDA_Chol3` como base por tener el fit más barato.

> **P12 · Implementar `TensorizedChol`. P14 · Implementar `EfficientChol`.**

**La idea.** Son las mismas dos optimizaciones de la Sección 2, pero ahora aplicadas a la variante Cholesky. Reusamos los *insights* ya demostrados:

- **`TensorizedChol`** (P12) = la idea de `TensorizedQDA` (apilar por clase, sacar el loop sobre clases) sobre la maquinaria de Cholesky. Hereda de `QDA_Chol3`, apila las $L^{-1}$ en `(k,p,p)` y calcula las $k$ verosimilitudes de una observación de un saque. Sigue iterando sobre observaciones.
- **`EfficientChol`** (P14) = la idea de `EfficientQDA` (sin `for`, sin $n\times n$) sobre Cholesky. Acá es aún más limpio: como la forma cuadrática ya es $\lVert y\rVert^2$ con $y = L^{-1}u$, la diagonal sale sola sumando $y\odot y$ sobre las features (`np.sum(y*y, axis=1)` → `(k,n)`). La $n\times n$ —que sería $y^Ty$— nunca se arma.

**El detalle.** El código de ambas está abajo. Es la combinación final: lo mejor de tensorización + lo mejor de Cholesky.

In [8]:
print(getsource(TensorizedChol))

class TensorizedChol(QDA_Chol3):
    """P12) Versión tensorizada de la variante Cholesky.

    Hereda de `QDA_Chol3` (que invierte la triangular L con `dtrtri`, la más
    barata en el fit). Apila las inversas triangulares L^-1 en un tensor
    (k, p, p) y las medias en (k, p, 1), igual que `TensorizedQDA`, para
    calcular las k log-condicionales de una observación en una sola pasada.

    Idea (P8): si Sigma = L L^T, entonces
        (x-mu)^T Sigma^-1 (x-mu) = || L^-1 (x-mu) ||^2 = (y**2).sum()
    con y = L^-1 (x-mu), y  0.5*log|Sigma^-1| = log( prod diag(L^-1) ).
    Paraleliza sobre clases (no sobre observaciones: sigue el for de predict).
    """

    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_inv = np.stack(self.L_invs)          # (k, p, p)
        self.tensor_means = np.stack(self.means)           # (k, p, 1)
        # 0.5*log|Sigma^-1| = sum(log(diag(L^-1))) por clase, precomputado: (k,)
        self.log_dets = np.log(
            np.

In [9]:
print(getsource(EfficientChol))

class EfficientChol(QDA_Chol3):
    """P14) Combina los insights de `EfficientQDA` y `TensorizedChol`:
    sin ciclo for y sin matriz de n x n.

    Para todas las observaciones a la vez (X de (p, n)):
        unbiased = X - mu_k            -> (k, p, n)
        y = L^-1 @ unbiased            -> (k, p, n)
        || y ||^2 por observación      = sum sobre p de y*y -> (k, n)
    La matriz n x n nunca aparece: en vez de y^T @ y (que sería (k, n, n))
    sumamos y*y a lo largo del eje de las p features.
    """

    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_inv = np.stack(self.L_invs)          # (k, p, p)
        self.tensor_means = np.stack(self.means)           # (k, p, 1)
        self.log_dets = np.log(
            np.diagonal(self.tensor_L_inv, axis1=1, axis2=2)
        ).sum(axis=1)                                      # (k,)

    def predict(self, X):
        # X: (p, n)
        unbiased = X[np.newaxis, :, :] - self.tensor_means  # (k, p, n)

### Verificación: las 9 variantes predicen idéntico

In [10]:
all_models = [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
              QDA_Chol1, QDA_Chol2, QDA_Chol3, TensorizedChol, EfficientChol]
ref = None
ok = True
for M in all_models:
    m = M(); m.fit(Xtr, ytr); p = m.predict(Xte)
    if ref is None: ref = p
    same = np.array_equal(p, ref); ok &= same
    print(f'{M.__name__:16s} igual a QDA? {same}')
print('\nTODAS idénticas:', ok)

QDA              igual a QDA? True
TensorizedQDA    igual a QDA? True
FasterQDA        igual a QDA? True
EfficientQDA     igual a QDA? True
QDA_Chol1        igual a QDA? True
QDA_Chol2        igual a QDA? True
QDA_Chol3        igual a QDA? True
TensorizedChol   igual a QDA? True
EfficientChol    igual a QDA? True

TODAS idénticas: True


### P13 — Benchmark de las 9 variantes (Wine)

In [11]:
for M in [TensorizedChol, EfficientChol]:
    b_wine.bench(M)

order9 = order7 + ['TensorizedChol', 'EfficientChol']
b_wine.summary(baseline='QDA').loc[order9, cols].round(4)

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,0.6917,1.0000,0.0077,1.0000,0.9826
TensorizedQDA,0.3175,2.1788,0.0121,0.6349,0.9851
FasterQDA,0.0185,37.4322,0.1093,0.0704,0.9848
EfficientQDA,0.0180,38.3400,0.0750,0.1026,0.9856
QDA_Chol1,0.3845,1.7989,0.0078,0.9843,0.9831
QDA_Chol2,0.8938,0.7739,0.0080,0.9660,0.9814
QDA_Chol3,0.3885,1.7805,0.0076,1.0099,0.9822
TensorizedChol,0.1556,4.4448,0.0124,0.6189,0.9826
EfficientChol,0.0123,56.2747,0.0603,0.1276,0.9836


### P13 (cont.) — Dataset grande: la asíntota de memoria

En Wine ($n$ chico, $k=3$) las diferencias de memoria casi no se ven. Repito el
benchmark en un subsample de **letters** ($k=26$, $p=16$), donde la matriz
$n\times n$ de `FasterQDA` pesa de verdad: con $n_{\text{test}}\approx 1000$ y $k=26$
son $(26,1000,1000)$ floats $\approx 208$ MB.

In [12]:
from numpy.random import RandomState
Xl, yl = get_letters_dataset(); yl = label_encode(yl.reshape(-1,1))
rs = RandomState(0); idx = rs.choice(len(Xl), 4000, replace=False)
Xl, yl = Xl[idx], yl[idx]
print('letters subsample:', Xl.shape, 'k =', len(np.unique(yl)))

b_let = Benchmark(Xl, yl, n_runs=25, warmup=5, mem_runs=10, test_sz=0.25, same_splits=False)
for M in all_models:
    b_let.bench(M)
b_let.summary(baseline='QDA').loc[order9, cols].round(4)

letters subsample: (4000, 16) k = 26
Benching params:
Total runs: 40
Warmup runs: 5
Peak Memory usage runs: 10
Running time runs: 25
Train size rows (approx): 3000
Test size rows (approx): 1000
Test size fraction: 0.25


QDA (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,107.8467,1.0000,0.0750,1.0000,0.8642
TensorizedQDA,22.1555,4.8677,0.1312,0.5718,0.8622
FasterQDA,2.8743,37.5206,204.8284,0.0004,0.8614
EfficientQDA,0.6709,160.7455,9.8368,0.0076,0.8595
QDA_Chol1,56.5913,1.9057,0.0725,1.0350,0.8618
QDA_Chol2,138.0085,0.7814,0.0726,1.0333,0.8606
QDA_Chol3,57.3562,1.8803,0.0723,1.0384,0.8635
TensorizedChol,3.5895,30.0451,0.1349,0.5563,0.8620
EfficientChol,0.5070,212.7154,9.8371,0.0076,0.8616


> **P13 · Comparar las 9 variantes. ¿Qué se observa? ¿Se condice con lo esperado?**

**Lectura conceptual.** Wine era demasiado chico para que se vieran las diferencias de fondo; el dataset grande las pone en evidencia y separa dos ejes que en chiquito parecían el mismo: **tiempo** y **memoria**.

- **En tiempo**, gana lo que ya anticipábamos: las versiones *Efficient* (`EfficientChol` ~210×, `EfficientQDA` ~160×). La mayor palanca sigue siendo eliminar el `for` de Python; Cholesky suma una mejora extra por tener la forma cuadrática más barata.
- **En memoria está el verdadero hallazgo del TP:** `FasterQDA` se dispara a **~205 MB** mientras `EfficientQDA`/`EfficientChol` se quedan en ~10 MB con el mismo o mejor tiempo. Acá se *ve* la matriz $n\times n$ de P4 — y se ve que la identidad de P5 no es un tecnicismo, sino lo que separa un modelo usable de uno que no escala.
- `QDA_Chol2` vuelve a quedar última en `predict` (resolver el sistema por observación), y las *Tensorized* quedan en el medio (paralelizan clases, no observaciones).

**¿Se condice con lo esperado?** Sí, y de forma más nítida que en Wine: el mejor modelo es el que **combina las tres ideas** —sin `for`, sin $n\times n$, con Cholesky—, es decir `EfficientChol`. Optimizar no fue elegir un truco, sino apilar varios sin perder corrección.

## 4. Accuracy por Cross-Validation (5-fold estratificado)

El `Benchmark` reporta `mean_accuracy` sobre repeated train/test splits aleatorios:
sirve para medir tiempo/memoria, pero como **métrica de calidad** es ruidosa (y con
`same_splits=False` cada modelo ve splits distintos). Para una estimación robusta uso
`StratifiedKFold`: cada observación se evalúa exactamente una vez como test y las
clases quedan balanceadas en cada fold. Como las 9 variantes predicen idéntico, la
accuracy de CV es la misma para todas — reporto el modelo y, de paso, confirmo que la
implementación no degrada la calidad.

In [13]:
for name, M in [('QDA', QDA), ('EfficientQDA', EfficientQDA), ('EfficientChol', EfficientChol)]:
    accs = cv_accuracy(M, X_full, y_enc, n_splits=5)
    print(f'{name:16s} CV acc = {accs.mean():.4f} +/- {accs.std():.4f}   folds={np.round(accs,3)}')

QDA              CV acc = 0.9887 +/- 0.0138   folds=[1.    0.972 1.    1.    0.971]
EfficientQDA     CV acc = 0.9887 +/- 0.0138   folds=[1.    0.972 1.    1.    0.971]
EfficientChol    CV acc = 0.9887 +/- 0.0138   folds=[1.    0.972 1.    1.    0.971]


## 5. Conclusiones

Más allá de los números, lo que deja el TP son algunas ideas que trascienden a QDA:

1. **Optimizar es, ante todo, encontrar dónde se va el tiempo — no calcular más rápido.** El gran salto no vino de aritmética más astuta sino de sacar los bucles de Python: el intérprete era el costo dominante, no las operaciones. Medir antes de optimizar es lo que reveló esto.

2. **Vectorizar "de una" puede esconder un costo cuadrático.** `FasterQDA` es la trampa elegante: corre rápido pero construye una matriz $n\times n$ de la que solo usa la diagonal. La lección es que *rápido* y *eficiente* son ejes distintos, y mirar solo el reloj te puede hacer elegir un modelo que no escala.

3. **Un poco de álgebra ahorra mucho cómputo.** La identidad $\operatorname{diag}(AB)=\sum_{\text{cols}}A\odot B^T$ (P5) y la factorización de Cholesky (P8) no son trucos de NumPy: son resultados matemáticos que bajan la complejidad de $O(n^2)$ a $O(np)$ y reemplazan una inversión general por una norma. Pensar la matemática *antes* de codear fue lo que habilitó las mejores variantes.

4. **Las optimizaciones se componen.** El mejor modelo, `EfficientChol`, no aplica una idea sino tres apiladas (sin `for` + sin $n\times n$ + Cholesky). Ninguna anula a las otras; se suman.

5. **Optimizar no es cambiar el modelo.** Las 9 variantes predicen *exactamente* lo mismo (verificado byte a byte) y la accuracy por cross-validation lo confirma. Toda la ganancia fue en *cómo* se computa, manteniendo intacto *qué* se computa — que es la única forma honesta de optimizar.